# PII Redaction with YAKE Keyword Extraction + Regex on SageMaker

This notebook demonstrates **keyword-based PII redaction** using John Snow Labs' Spark NLP (`YakeKeywordExtraction`) combined with **regex patterns** for structured PII (SSN, dates, phone numbers) on Amazon SageMaker.

**Approach:**
- `YakeKeywordExtraction` — unsupervised statistical keyword extractor that surfaces names, organisations, and domain-specific terms without needing a labelled NER model
- Regex pre-pass — catches structured PII (SSN `\d{3}-\d{2}-\d{4}`, dates, phone numbers, postal codes) that YAKE may miss because they are not "keywords"
- All detected spans replaced with `[REDACTED]`

**Why YAKE over NER?**
- The `ner_dl` model (CoNLL, 4-class) misses DATE, TIME, and numeric PII
- YAKE is language-agnostic and requires no pre-trained label set
- Both approaches are complementary — this notebook combines them

**SageMaker v2.x compatible** — uses `sagemaker>=2.0` SDK APIs.

## 1. Environment Setup

In [14]:
import subprocess, sys, os

java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

Java already available: openjdk version "11.0.30" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [15]:
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "spark-nlp" \
    "spark-nlp-display" \
    "boto3"

print("All packages installed.")

All packages installed.


## 2. SageMaker v2.x Session Setup

In [16]:
import boto3
import sagemaker
from sagemaker import get_execution_role

boto_session = boto3.Session()
sm_session   = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region = boto_session.region_name or sm_session.boto_region_name
bucket = sm_session.default_bucket()   # assigned before any wildcard imports
prefix = "spark-nlp/redaction-yake"

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Default S3 bucket     : {bucket}")
print(f"Execution role ARN    : {role}")

SageMaker SDK version : 2.257.1
Region                : us-west-2
Default S3 bucket     : sagemaker-us-west-2-493644444178
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd


## 3. Import Spark NLP & Start Spark Session

> **Note:** `pyspark.sql.functions` contains a built-in `bucket()` function.
> Always use `import pyspark.sql.functions as F` (alias) rather than `from pyspark.sql.functions import *`
> to avoid silently overwriting the `bucket` string variable assigned in the cell above.

In [17]:
import sys
print("Python executable:", sys.executable)

import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline

import pyspark.sql.functions as F
from pyspark.sql.types import StringType, ArrayType

import pandas as pd
import re
import warnings

warnings.filterwarnings("ignore")
print("Spark NLP version:", sparknlp.version())

Python executable: /home/cjen/mySageMaker/ML/spark-nlp/redaction/.venv/bin/python
Spark NLP version: 6.3.3


In [18]:
params = {
    "spark.driver.memory"           : "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize"    : "2000M",
    "spark.ui.enabled"              : "false",
}

spark = sparknlp.start(params=params)
spark

## 4. Sample Data

Same texts as `redaction_sagemaker.ipynb` — they contain names, dates, SSNs, addresses, and IDs that the NER-only approach missed.

In [19]:
sample_texts = [
    "Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.",
    "Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.",
    "The contract signed by Alice Wong and Bob Martin on January 15 2023 includes a clause for New York jurisdiction.",
    "Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 on 09/22/2024 at 3:45 PM.",
    "Invoice #INV-2024-88 issued to Acme Corp, attention David Brown, 100 King Street West, Vancouver, BC V6B 1A1.",
]

pdf      = pd.DataFrame(sample_texts, columns=["text"])
spark_df = spark.createDataFrame(pdf)
spark_df.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+
|text                                                                                                              |
+------------------------------------------------------------------------------------------------------------------+
|Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.|
|Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.     |
|The contract signed by Alice Wong and Bob Martin on January 15 2023 includes a clause for New York jurisdiction.  |
|Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 on 09/22/2024 at 3:45 PM.            |
|Invoice #INV-2024-88 issued to Acme Corp, attention David Brown, 100 King Street West, Vancouver, BC V6B 1A1.     |
+---------------------------------------------------------------

## 5. Regex Pre-pass — Structured PII

YAKE is a statistical keyword extractor — it will surface names and organisations but is unlikely to flag numeric patterns such as SSNs, ISO dates, or phone numbers because these have low term-frequency scores.

The regex pre-pass catches:
| Pattern | Example |
|---|---|
| SSN | `123-45-6789` |
| ISO date | `2024-11-01` |
| US/CA date | `09/22/2024` |
| Written date | `March 4 1975`, `January 15 2023` |
| Time | `3:45 PM` |
| Postal code | `V6B 1A1` |
| Phone | `(416) 555-0123` |
| Employee/complaint IDs | `E-4892`, `CR-00312`, `INV-2024-88` |

In [20]:
# Ordered from most-specific to least-specific to avoid partial overlaps
REGEX_PII_PATTERNS = [
    # SSN  (must come before plain numbers)
    (r'\b\d{3}-\d{2}-\d{4}\b',                          '[SSN]'),
    # ISO date  2024-11-01
    (r'\b\d{4}-\d{2}-\d{2}\b',                          '[DATE]'),
    # US/CA date  09/22/2024  or  9/1/2024
    (r'\b\d{1,2}/\d{1,2}/\d{4}\b',                      '[DATE]'),
    # Written date with 4-digit year  March 4 1975 / January 15 2023
    (r'\b(?:January|February|March|April|May|June|July|August|September|'
     r'October|November|December)\s+\d{1,2}\s+\d{4}\b', '[DATE]'),
    # Time  3:45 PM / 14:30
    (r'\b\d{1,2}:\d{2}(?:\s?[AP]M)?\b',                 '[TIME]'),
    # Canadian postal code  V6B 1A1
    (r'\b[A-Z]\d[A-Z]\s?\d[A-Z]\d\b',                  '[POSTAL_CODE]'),
    # Phone  (416) 555-0123 / 416-555-0123 / +1-800-555-0123
    (r'(?:\+?1[-.])?\(?\d{3}\)?[-.]\d{3}[-.]\d{4}\b',   '[PHONE]'),
    # Alphanumeric IDs: E-4892, CR-00312, INV-2024-88
    (r'\b[A-Z]{1,4}-[A-Z0-9]{2,}-?[0-9]{0,4}\b',       '[ID]'),
]

def regex_redact(text):
    """Apply all regex PII patterns left-to-right."""
    if not text:
        return text
    for pattern, replacement in REGEX_PII_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text

regex_redact_udf = F.udf(regex_redact, StringType())

# Apply regex redaction first
regex_df = spark_df.withColumn("regex_redacted", regex_redact_udf(F.col("text")))
regex_df.select("text", "regex_redacted").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------+
|text                                                                                                              |regex_redacted                                                                                              |
+------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------+
|Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.|Patient John Smith, born on [DATE], was admitted to Montreal General Hospital on [DATE] with chest pain.    |
|Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple St

## 6. YAKE Keyword Extraction Pipeline

`YakeKeywordExtraction` is an **unsupervised, language-agnostic** keyword extractor.
It scores terms by position, frequency, and co-occurrence — no pre-trained weights needed.

Key parameters:
| Parameter | Default | Effect |
|---|---|---|
| `setMinNGrams` | 1 | Minimum n-gram size |
| `setMaxNGrams` | 3 | Maximum n-gram size (catches multi-word names) |
| `setNKeywords` | 20 | Max keywords per document |
| `setThreshold` | 1.0 | Lower = stricter (fewer, higher-quality keywords) |
| `setWindowSize` | 3 | Context window for co-occurrence scoring |

In [21]:
# ── Stage 1: Raw text → Document ─────────────────────────────────────────────
document_assembler = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

# ── Stage 2: Sentence segmentation ───────────────────────────────────────────
sentence_detector = (
    SentenceDetector()
    .setInputCols(["document"])
    .setOutputCol("sentence")
)

# ── Stage 3: Tokenisation ─────────────────────────────────────────────────────
tokenizer = (
    Tokenizer()
    .setInputCols(["sentence"])
    .setOutputCol("token")
)

# ── Stage 4: YAKE keyword extraction ─────────────────────────────────────────
# setThreshold: keywords with score BELOW threshold are kept.
# Lower threshold → stricter, fewer keywords (names/orgs rank well).
# Raise to ~2.0 if you want broader coverage.
yake = (
    YakeKeywordExtraction()
    .setInputCols(["token"])
    .setOutputCol("keywords")
    .setMinNGrams(1)
    .setMaxNGrams(3)
    .setNKeywords(30)
    .setThreshold(1.5)
    .setWindowSize(3)
)

yake_pipeline = Pipeline(
    stages=[
        document_assembler,
        sentence_detector,
        tokenizer,
        yake,
    ]
)

print("YAKE pipeline assembled.")

YAKE pipeline assembled.


In [22]:
empty_df   = spark.createDataFrame([[""]], ["text"])
yake_model = yake_pipeline.fit(empty_df)
print("Model ready.")

Model ready.


## 7. Run YAKE Inference & Inspect Extracted Keywords

In [23]:
result_df = yake_model.transform(spark_df)

# Deduplicate keywords per row and attach to the dataframe
result_df = result_df.withColumn(
    "unique_keywords", F.array_distinct(F.col("keywords.result"))
)

# Inspect extracted keywords per sentence
keywords_flat = result_df.select(
    F.col("text"),
    F.explode("keywords").alias("kw")
).select(
    "text",
    F.col("kw.result").alias("keyword"),
    F.col("kw.metadata")["score"].cast("double").alias("score"),
).orderBy("text", "score")

keywords_flat.show(60, truncate=80)

+--------------------------------------------------------------------------------+-------------------------+-------------------+
|                                                                            text|                  keyword|              score|
+--------------------------------------------------------------------------------+-------------------------+-------------------+
|Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 ...|                sarah lee| 0.5066323531331214|
|Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 ...|                    sarah| 0.5798862558280943|
|Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 ...|                      lee| 0.5798862558280943|
|Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 ...|                       id| 0.5798862558280943|
|Call center agent Sarah Lee (employee ID E-4892) handled complaint #CR-00312 ...|               

## 8. Apply YAKE Redaction

We replace each extracted keyword with `[REDACTED]` using case-insensitive whole-word regex substitution (same pattern as the `highlight_udf` in the fork notebook, but replacing instead of wrapping in `<span>`).

This is then **composed with the regex pre-pass** from Section 5 to cover both structured PII (dates, SSNs) and unstructured PII (names, organisations).

In [24]:
def yake_redact(text, keywords):
    """
    Replace each YAKE keyword in text with [REDACTED].
    Uses whole-word, case-insensitive regex — same logic as the
    highlight_udf in fork-of-tc-lexicon-visualization-spark-nlp-dec2025.ipynb
    but replaces instead of highlighting.
    """
    if not text or not keywords:
        return text
    # Sort by length descending so longer phrases are matched before
    # their component words (e.g. "John Smith" before "John")
    for kw in sorted(keywords, key=len, reverse=True):
        text = re.sub(
            r'(\b%s\b)' % re.escape(kw),
            '[REDACTED]',
            text,
            flags=re.IGNORECASE,
        )
    return text

yake_redact_udf = F.udf(yake_redact, StringType())

# ── Compose: regex pre-pass  →  YAKE redaction ───────────────────────────────
redacted_df = (
    result_df
    # Step 1: redact structured PII with regex
    .withColumn("step1_regex", regex_redact_udf(F.col("text")))
    # Step 2: redact YAKE keywords from the regex-cleaned text
    .withColumn("redacted_text", yake_redact_udf(F.col("step1_regex"), F.col("unique_keywords")))
    .select("text", "unique_keywords", "step1_regex", "redacted_text")
)

redacted_df.select("text", "redacted_text").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------+
|text                                                                                                              |redacted_text                                                                                                         |
+------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------+
|Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.|[REDACTED], [REDACTED] on [DATE], was [REDACTED] to [REDACTED] on [DATE] with [REDACTED].                             |
|Dr. Emily Carter reviewed the case of Michael Johnson, 

## 9. Side-by-Side: Original → Regex-only → YAKE+Regex

In [25]:
rows = redacted_df.collect()
for r in rows:
    print("ORIGINAL  :", r["text"])
    print("REGEX ONLY:", r["step1_regex"])
    print("FULL      :", r["redacted_text"])
    print("KEYWORDS  :", r["unique_keywords"])
    print("-" * 110)

ORIGINAL  : Patient John Smith, born on March 4 1975, was admitted to Montreal General Hospital on 2024-11-01 with chest pain.
REGEX ONLY: Patient John Smith, born on [DATE], was admitted to Montreal General Hospital on [DATE] with chest pain.
FULL      : [REDACTED], [REDACTED] on [DATE], was [REDACTED] to [REDACTED] on [DATE] with [REDACTED].
KEYWORDS  : ['patient', 'john', 'smith', 'born', 'march', 'admitted', 'montreal', 'general', 'hospital', 'chest', 'pain', 'patient john', 'john smith', 'montreal general', 'general hospital', 'chest pain', 'patient john smith', 'born on march', 'admitted to montreal', 'montreal general hospital']
--------------------------------------------------------------------------------------------------------------
ORIGINAL  : Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.
REGEX ONLY: Dr. Emily Carter reviewed the case of Michael Johnson, SSN [SSN], residing at 42 Maple Street, Toronto.
FULL   

## 10. HTML Visualization

Because `NerVisualizer` expects `ner_chunk` annotations with an `entity` metadata key, it cannot render YAKE `keywords` output directly.

Instead we build a two-layer inline highlight using the same `<span>` approach as `fork-of-tc-lexicon-visualization-spark-nlp-dec2025.ipynb`:

| Colour | Meaning |
|---|---|
| 🟠 Orange `#FFB06E` | Regex-detected structured PII (dates, SSN, IDs, times, postal codes) |
| 🟢 Green `#90EE90` | YAKE-extracted keyword (names, organisations, domain terms) |

The `LightPipeline` is used for fast single-row annotation (no Spark shuffle overhead).

In [26]:
from IPython.display import display, HTML

# ── LightPipeline for fast single-row annotation ─────────────────────────────
light_model = LightPipeline(yake_model)

# ── Compile regex patterns once for the visualiser ───────────────────────────
_COMPILED_REGEX = [
    (re.compile(pattern, re.IGNORECASE), label)
    for pattern, label in REGEX_PII_PATTERNS
]

def _build_html(text):
    """
    Returns an HTML string with two layers of inline highlighting:
      - Orange: spans matched by the regex PII patterns
      - Green : YAKE keywords (extracted via LightPipeline)
    Mirrors the NerVisualizer colour-band style.
    """
    # ── Collect regex spans ───────────────────────────────────────────────────
    regex_spans = []  # list of (start, end, label)
    for compiled, label in _COMPILED_REGEX:
        for m in compiled.finditer(text):
            # Skip if already covered by a wider match
            if not any(s <= m.start() and m.end() <= e for s, e, _ in regex_spans):
                regex_spans.append((m.start(), m.end(), label))

    # ── Collect YAKE keyword spans ────────────────────────────────────────────
    annotations = light_model.fullAnnotate(text)[0]
    seen_kws = set()
    yake_spans = []  # list of (start, end, keyword)
    # Sort keywords longest-first so multi-word phrases are matched first
    keywords_sorted = sorted(
        annotations["keywords"],
        key=lambda k: len(k.result),
        reverse=True,
    )
    for kw in keywords_sorted:
        kw_text = kw.result
        if kw_text.lower() in seen_kws:
            continue
        seen_kws.add(kw_text.lower())
        for m in re.finditer(r'(\b%s\b)' % re.escape(kw_text), text, re.IGNORECASE):
            # Skip if span is already claimed by a regex PII match
            if not any(s <= m.start() and m.end() <= e for s, e, _ in regex_spans):
                yake_spans.append((m.start(), m.end(), kw_text))

    # ── Merge and sort all spans by start position ────────────────────────────
    all_spans = (
        [(s, e, lbl, "regex") for s, e, lbl in regex_spans] +
        [(s, e, kw,  "yake")  for s, e, kw  in yake_spans]
    )
    # Remove overlapping spans (keep first-encountered after sort by start)
    all_spans.sort(key=lambda x: x[0])
    merged = []
    last_end = 0
    for span in all_spans:
        if span[0] >= last_end:
            merged.append(span)
            last_end = span[1]

    # ── Build HTML ────────────────────────────────────────────────────────────
    STYLE = {
        "regex": ("background-color:#FFB06E; border-radius:3px; padding:1px 3px;"
                  " font-weight:bold; font-size:0.9em;"),
        "yake":  ("background-color:#90EE90; border-radius:3px; padding:1px 3px;"
                  " font-size:0.9em;"),
    }
    LABEL_STYLE = "font-size:0.7em; color:#555; vertical-align:super; margin-left:2px;"

    parts = []
    cursor = 0
    for start, end, label, kind in merged:
        parts.append(text[cursor:start])
        parts.append(
            f'<span style="{STYLE[kind]}">'
            f'{text[start:end]}'
            f'<span style="{LABEL_STYLE}">[{label.upper()}]</span>'
            f'</span>'
        )
        cursor = end
    parts.append(text[cursor:])

    body = "".join(parts)
    return (
        '<div style="font-family:monospace; font-size:1em; line-height:1.8;'
        ' border:1px solid #ddd; border-radius:6px; padding:10px 14px;'
        f' margin-bottom:10px; background:#fafafa;">{body}</div>'
    )


# ── Legend ────────────────────────────────────────────────────────────────────
legend_html = (
    '<div style="font-family:sans-serif; font-size:0.85em; margin-bottom:12px;">'
    '<b>Legend:</b>&nbsp;'
    '<span style="background:#FFB06E; border-radius:3px; padding:1px 6px;">Regex PII</span>'
    '&nbsp;&nbsp;'
    '<span style="background:#90EE90; border-radius:3px; padding:1px 6px;">YAKE keyword</span>'
    '</div>'
)
display(HTML(legend_html))

# ── Render each sample sentence ───────────────────────────────────────────────
for text in sample_texts:
    display(HTML(_build_html(text)))

## 10. Threshold Sensitivity Analysis

YAKE scores are **lower = more relevant**. The `setThreshold` parameter filters out keywords whose score is above the threshold.
The table below shows how coverage changes with different thresholds.

In [27]:
light_model = LightPipeline(yake_model)

thresholds = [0.5, 1.0, 1.5, 2.5]
sample = sample_texts[1]  # SSN + names text

print(f"Text: {sample}\n")
for t in thresholds:
    annotations = light_model.fullAnnotate(sample)[0]
    # Filter keywords by threshold manually for comparison
    kws = [
        k.result
        for k in annotations["keywords"]
        if float(k.metadata["score"]) <= t
    ]
    kws = list(dict.fromkeys(kws))  # deduplicate, preserve order
    print(f"  threshold={t:<4}  {len(kws):>2} keywords: {kws}")

Text: Dr. Emily Carter reviewed the case of Michael Johnson, SSN 123-45-6789, residing at 42 Maple Street, Toronto.

  threshold=0.5   13 keywords: ['emily', 'carter', 'michael', 'johnson', 'ssn', 'maple', 'street', 'toronto', 'emily carter', 'carter reviewed', 'michael johnson', 'maple street', 'emily carter reviewed']
  threshold=1.0   18 keywords: ['dr', 'emily', 'carter', 'reviewed', 'case', 'michael', 'johnson', 'ssn', 'residing', 'maple', 'street', 'toronto', 'emily carter', 'carter reviewed', 'michael johnson', 'maple street', 'emily carter reviewed', 'case of michael']
  threshold=1.5   19 keywords: ['dr', 'emily', 'carter', 'reviewed', 'case', 'michael', 'johnson', 'ssn', 'residing', 'maple', 'street', 'toronto', 'emily carter', 'carter reviewed', 'michael johnson', 'maple street', 'emily carter reviewed', 'reviewed the case', 'case of michael']
  threshold=2.5   19 keywords: ['dr', 'emily', 'carter', 'reviewed', 'case', 'michael', 'johnson', 'ssn', 'residing', 'maple', 'stree

## 11. (Optional) Save Redacted Output to S3

Uses the **SageMaker v2.x** `sagemaker.s3.S3Uploader` API.

In [28]:
import tempfile
from sagemaker.s3 import S3Uploader

local_output = os.path.join(tempfile.mkdtemp(), "redacted_yake_output.csv")
redacted_df.select("text", "redacted_text").toPandas().to_csv(local_output, index=False)

s3_uri = f"s3://{bucket}/{prefix}/redacted_yake_output.csv"

S3Uploader.upload(
    local_path=local_output,
    desired_s3_uri=s3_uri,
    sagemaker_session=sm_session,
)
print(f"Redacted output uploaded to: {s3_uri}")

Redacted output uploaded to: s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction-yake/redacted_yake_output.csv


## 12. Cleanup

In [29]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
